# AutoBI — Step 3B: End-to-End ETL Pipeline & Batch Log Demo (SYNTHETIC)

This notebook runs `core.etl.run_etl_pipeline` on the **SYNTHETIC** sample datasets (`SYNTHETIC_sales.csv` and `SYNTHETIC_hr.xlsx`), inspects the curated tables (`dim_date`, relationships, reconciliation, lineage), demonstrates idempotent reruns, and prints the DuckDB `etl_batch_log` table.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "core").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import Settings
from core.etl import run_etl_pipeline
from core.schemas import CleaningPlan
from core.storage import list_duckdb_tables, query_duckdb, read_etl_batch_log

settings = Settings(_env_file=None, data_dir=PROJECT_ROOT / "data", output_dir=PROJECT_ROOT / "output")
sample_files = [
    PROJECT_ROOT / "sample_data" / "SYNTHETIC_sales.csv",
    PROJECT_ROOT / "sample_data" / "SYNTHETIC_hr.xlsx",
]

plans = {
    "synthetic_sales": CleaningPlan(
        table_name="synthetic_sales",
        description="Normalize region casing and drop constant tag",
        operations=[
            {"op": "trim_text", "column": "region"},
            {"op": "standardize_case", "column": "region", "case": "title"},
            {"op": "drop_column", "column": "data_origin"},
        ],
    )
}

result = run_etl_pipeline(sample_files, cleaning_plans=plans, settings=settings, actor="notebook_demo")
print(f"Batch ID: {result.batch_id} | Validation passed: {result.validation_report.passed}")
print("Curated DuckDB tables:", list_duckdb_tables(result.curated_duckdb_path))

In [ ]:
print("=== Reconciliation ===")
for recon in result.reconciliations:
    print(
        f"{recon.table_name}: read={recon.rows_read}, staged={recon.rows_staged}, "
        f"curated={recon.rows_curated}, rejected={recon.rows_rejected}, "
        f"dups={recon.rows_dropped_duplicates}, reconciled={recon.is_reconciled}"
    )

print("\n=== Curated Relationships ===")
for rel in result.relationships:
    print(f"  {rel.from_table}.{rel.from_column} -> {rel.to_table}.{rel.to_column} ({rel.cardinality}, {rel.source})")

In [ ]:
# Idempotent rerun produces zero duplicates and reuses the exact verified batch
replay = run_etl_pipeline(sample_files, cleaning_plans=plans, settings=settings, actor="notebook_demo")
print(f"Second run idempotent_replay={replay.idempotent_replay}, same batch_id={replay.batch_id == result.batch_id}")

print("\n=== Curated dim_date Preview ===")
print(query_duckdb(result.curated_duckdb_path, "SELECT * FROM dim_date LIMIT 5"))

print("\n=== ETL Batch Log Table ===")
batch_log_df = read_etl_batch_log(settings=settings)
print(batch_log_df.to_string(index=False))